# 🌦️ Weather Recognition — Model Evaluation & Analysis

This notebook loads all 5 trained models, evaluates them on the held-out test set, and produces a comparative analysis.

**Models covered:** AlexNet · VGG16 · ResNet50 · MobileNetV2 · ViT-B/16

---

In [ ]:
# ═══════════════════════════════════════════════════════════════
# Environment Detection — runs identically on Colab and locally
# ═══════════════════════════════════════════════════════════════
import os, sys

# Detect Colab
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    # ── Mount Google Drive ──────────────────────────────────────
    from google.colab import drive
    drive.mount('/content/drive')

    # ── ✏️  Set this to wherever your folder lives in Drive ─────
    DRIVE_ROOT  = '/content/drive/MyDrive/Recognition_Weather'

    PROJECT_ROOT = DRIVE_ROOT
    DATASET_DIR  = os.path.join(DRIVE_ROOT, 'dataset')
    MODELS_DIR   = os.path.join(DRIVE_ROOT, 'models')

    # Install extra deps that Colab doesn't have by default
    os.system('pip install keras-hub tokenizers sentencepiece opencv-python-headless -q')

else:
    # ── Local machine (Mac / Linux) ─────────────────────────────
    NOTEBOOK_DIR = os.path.abspath('')
    PROJECT_ROOT = os.path.abspath(os.path.join(NOTEBOOK_DIR, '..'))
    DATASET_DIR  = os.path.join(PROJECT_ROOT, 'dataset')
    MODELS_DIR   = os.path.join(PROJECT_ROOT, 'models')

# Make project root importable (for utils.py etc.)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f'Environment  : {"Google Colab" if IN_COLAB else "Local"}')
print(f'Project root : {PROJECT_ROOT}')
print(f'Dataset dir  : {DATASET_DIR}')
print(f'Models dir   : {MODELS_DIR}')
print(f'Dataset OK   : {os.path.isdir(DATASET_DIR)}')
print(f'Models OK    : {os.path.isdir(MODELS_DIR)}')


## Section 1 — Setup & Imports

In [ ]:
import os
import sys
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
import keras_hub

from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, BatchNormalization,
    Flatten, Dense, Dropout, GlobalAveragePooling2D, LayerNormalization
)
from tensorflow.keras.applications import VGG16, ResNet50, MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, accuracy_score
)

# Paths set by the Environment Detection cell above
# (PROJECT_ROOT, DATASET_DIR, MODELS_DIR are already defined)

DATASET_DIR = os.path.join(PROJECT_ROOT, 'dataset')
MODELS_DIR  = os.path.join(PROJECT_ROOT, 'models')

# ── Constants
IMAGE_SIZE  = (224, 224)
BATCH_SIZE  = 32
RANDOM_SEED = 42

print(f'TensorFlow : {tf.__version__}')
print(f'Dataset dir: {DATASET_DIR}')
print(f'Models dir : {MODELS_DIR}')
print(f'GPU        : {bool(tf.config.list_physical_devices("GPU"))}')

## Section 2 — Dataset Preparation

In [ ]:
def load_dataset(dataset_dir):
    records = []
    for label in sorted(os.listdir(dataset_dir)):
        class_dir = os.path.join(dataset_dir, label)
        if not os.path.isdir(class_dir):
            continue
        for fname in os.listdir(class_dir):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                records.append({'filename': os.path.join(class_dir, fname), 'label': label})
    return pd.DataFrame(records)

df = load_dataset(DATASET_DIR)

# ── Drop any class with fewer than 2 images (stratify requires ≥2 per class)
class_counts = df['label'].value_counts()
small_classes = class_counts[class_counts < 2].index.tolist()
if small_classes:
    print(f'WARNING: Dropping classes with < 2 images: {small_classes}')
    df = df[~df['label'].isin(small_classes)].reset_index(drop=True)

train_df, test_df = train_test_split(
    df, test_size=0.2, stratify=df['label'], random_state=RANDOM_SEED
)

NUM_CLASSES = df['label'].nunique()
CLASS_NAMES = sorted(df['label'].unique().tolist())

print(f'Total  : {len(df)}')
print(f'Train  : {len(train_df)}')
print(f'Test   : {len(test_df)}')
print(f'Classes ({NUM_CLASSES}): {CLASS_NAMES}')

dist = df['label'].value_counts().sort_index().rename_axis('Class').reset_index(name='Total')
dist['Train'] = train_df['label'].value_counts().sort_index().values
dist['Test']  = test_df['label'].value_counts().sort_index().values
display(dist.style.set_caption('Dataset Distribution'))


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(CLASS_NAMES))
w = 0.35
ax.bar(x - w/2, dist['Train'], w, label='Train', color='steelblue')
ax.bar(x + w/2, dist['Test'],  w, label='Test',  color='coral')
ax.set_xticks(x)
ax.set_xticklabels(CLASS_NAMES, rotation=30, ha='right')
ax.set_ylabel('Count')
ax.set_title('Train / Test Image Distribution per Class')
ax.legend()
plt.tight_layout()
plt.show()

## Section 3 — Model Reconstruction & Weight Loading

Each architecture is rebuilt exactly as defined in the training scripts, then saved weights are loaded.

> **Note:** ViT requires `keras_hub` to download/cache the backbone on first run.

In [ ]:
# ── 3.1 AlexNet
from tensorflow.keras.applications.vgg16 import preprocess_input as alexnet_preprocess

class AlexNetClassifier(Model):
    def __init__(self, num_classes, dropout_rate=0.5):
        super().__init__()
        self.conv1 = Conv2D(96,  (11,11), strides=4, activation='relu', padding='valid')
        self.bn1   = BatchNormalization()
        self.pool1 = MaxPooling2D((3,3), strides=2)
        self.conv2 = Conv2D(256, (5,5),  strides=1, activation='relu', padding='same')
        self.bn2   = BatchNormalization()
        self.pool2 = MaxPooling2D((3,3), strides=2)
        self.conv3 = Conv2D(384, (3,3),  strides=1, activation='relu', padding='same')
        self.bn3   = BatchNormalization()
        self.conv4 = Conv2D(384, (3,3),  strides=1, activation='relu', padding='same')
        self.bn4   = BatchNormalization()
        self.conv5 = Conv2D(256, (3,3),  strides=1, activation='relu', padding='same')
        self.bn5   = BatchNormalization()
        self.pool5 = MaxPooling2D((3,3), strides=2)
        self.flatten = Flatten()
        self.dense1  = Dense(4096, activation='relu')
        self.drop1   = Dropout(dropout_rate)
        self.dense2  = Dense(4096, activation='relu')
        self.drop2   = Dropout(dropout_rate)
        self.out     = Dense(num_classes, activation='softmax')

    def call(self, x, training=False):
        x = self.pool1(self.bn1(self.conv1(x), training=training))
        x = self.pool2(self.bn2(self.conv2(x), training=training))
        x = self.bn3(self.conv3(x), training=training)
        x = self.bn4(self.conv4(x), training=training)
        x = self.pool5(self.bn5(self.conv5(x), training=training))
        x = self.flatten(x)
        x = self.drop1(self.dense1(x), training=training)
        x = self.drop2(self.dense2(x), training=training)
        return self.out(x)

model_alexnet = AlexNetClassifier(num_classes=NUM_CLASSES)
model_alexnet.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model_alexnet(tf.zeros([1, 224, 224, 3]))
model_alexnet.load_weights(os.path.join(MODELS_DIR, 'best_alexnet.weights.h5'))
print(f'AlexNet params     : {model_alexnet.count_params():,}')
print('AlexNet weights loaded OK')

In [ ]:
# ── 3.2 VGG16
from tensorflow.keras.applications.vgg16 import preprocess_input as vgg16_preprocess

class VGG16Classifier(Model):
    def __init__(self, num_classes, dropout_rate=0.5):
        super().__init__()
        self.base   = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
        self.base.trainable = True
        self.pool   = GlobalAveragePooling2D()
        self.bn1    = BatchNormalization()
        self.dense1 = Dense(512, activation='relu')
        self.drop1  = Dropout(dropout_rate)
        self.bn2    = BatchNormalization()
        self.dense2 = Dense(256, activation='relu')
        self.drop2  = Dropout(dropout_rate)
        self.out    = Dense(num_classes, activation='softmax')

    def call(self, x, training=False):
        x = self.base(x, training=False)
        x = self.pool(x)
        x = self.drop1(self.dense1(self.bn1(x, training=training)), training=training)
        x = self.drop2(self.dense2(self.bn2(x, training=training)), training=training)
        return self.out(x)

model_vgg = VGG16Classifier(num_classes=NUM_CLASSES)
model_vgg.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model_vgg(tf.zeros([1, 224, 224, 3]))
model_vgg.load_weights(os.path.join(MODELS_DIR, 'best_vgg16.weights.h5'))
print(f'VGG16 params       : {model_vgg.count_params():,}')
print('VGG16 weights loaded OK')

In [ ]:
# ── 3.3 ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input as resnet_preprocess

def create_resnet_model(num_classes):
    base = ResNet50(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
    base.trainable = True
    x   = GlobalAveragePooling2D()(base.output)
    x   = BatchNormalization()(x)
    x   = Dense(256, activation='relu')(x)
    x   = Dropout(0.5)(x)
    out = Dense(num_classes, activation='softmax')(x)
    return Model(inputs=base.input, outputs=out)

model_resnet = create_resnet_model(NUM_CLASSES)
model_resnet.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model_resnet.load_weights(os.path.join(MODELS_DIR, 'best_resnet.weights.h5'))
print(f'ResNet50 params    : {model_resnet.count_params():,}')
print('ResNet50 weights loaded OK')

In [ ]:
# ── 3.4 MobileNetV2
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess

def create_mobilenet_model(num_classes):
    base = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3), alpha=1.0)
    base.trainable = True
    x   = GlobalAveragePooling2D()(base.output)
    x   = Dense(128, activation='relu')(x)
    x   = Dropout(0.3)(x)
    out = Dense(num_classes, activation='softmax')(x)
    return Model(inputs=base.input, outputs=out)

model_mobilenet = create_mobilenet_model(NUM_CLASSES)
model_mobilenet.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model_mobilenet.load_weights(os.path.join(MODELS_DIR, 'best_mobilenet.weights.h5'))
print(f'MobileNetV2 params : {model_mobilenet.count_params():,}')
print('MobileNetV2 weights loaded OK')

In [ ]:
# ── 3.5 ViT-B/16
PRESET = 'vit_base_patch16_224_imagenet21k'

class ViTClassifier(Model):
    def __init__(self, num_classes, dropout_rate=0.3, **kwargs):
        super().__init__(**kwargs)
        self.backbone     = keras_hub.models.ViTBackbone.from_preset(PRESET)
        self.backbone.trainable = True
        self.norm         = LayerNormalization(epsilon=1e-6)
        self.dense1       = Dense(256, activation='gelu')
        self.drop         = Dropout(dropout_rate)
        self.out          = Dense(num_classes, activation='softmax')
        self.num_classes  = num_classes
        self.dropout_rate = dropout_rate

    def call(self, x, training=False):
        x   = self.backbone(x, training=False)
        cls = x[:, 0, :]
        cls = self.norm(cls)
        cls = self.dense1(cls)
        cls = self.drop(cls, training=training)
        return self.out(cls)

    def get_config(self):
        c = super().get_config()
        c.update({'num_classes': self.num_classes, 'dropout_rate': self.dropout_rate})
        return c

print('Loading ViT backbone (may download on first run)...')
model_vit = ViTClassifier(num_classes=NUM_CLASSES)
model_vit.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model_vit(tf.zeros([1, 224, 224, 3]))
model_vit.load_weights(os.path.join(MODELS_DIR, 'best_vit.weights.h5'))
print(f'ViT-B/16 params    : {model_vit.count_params():,}')
print('ViT weights loaded OK')

## Section 4 — Per-Model Evaluation

In [ ]:
# ── Shared helpers
def make_test_gen(preprocess_fn):
    datagen = ImageDataGenerator(preprocessing_function=preprocess_fn)
    return datagen.flow_from_dataframe(
        dataframe=test_df,
        x_col='filename', y_col='label',
        target_size=IMAGE_SIZE,
        color_mode='rgb',
        class_mode='categorical',
        batch_size=BATCH_SIZE,
        shuffle=False,
        verbose=0
    )

def rescale(x):
    return x / 255.0

def evaluate_model(name, model, test_gen):
    sep = '=' * 60
    print()
    print(sep)
    print('  ' + name)
    print(sep)

    # Predictions
    test_gen.reset()
    probs     = model.predict(test_gen, verbose=0)
    preds     = np.argmax(probs, axis=1)
    labels    = test_gen.class_indices
    idx2cls   = {v: k for k, v in labels.items()}
    pred_classes = [idx2cls[p] for p in preds]
    true_classes = test_df['label'].tolist()

    # Loss / accuracy
    test_gen.reset()
    loss, acc = model.evaluate(test_gen, verbose=0)
    print(f'Test Loss     : {loss:.4f}')
    print(f'Test Accuracy : {acc*100:.2f}%')

    # Classification report
    print('\nClassification Report:')
    print(classification_report(true_classes, pred_classes, target_names=CLASS_NAMES))

    macro_f1    = f1_score(true_classes, pred_classes, average='macro')
    weighted_f1 = f1_score(true_classes, pred_classes, average='weighted')

    # Confusion matrix
    cm = confusion_matrix(true_classes, pred_classes, labels=CLASS_NAMES)
    fig, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(name + ' — Confusion Matrix')
    plt.tight_layout()
    plt.show()

    # Sample prediction grid (12 images)
    fig, axes = plt.subplots(3, 4, figsize=(14, 9),
                             subplot_kw={'xticks': [], 'yticks': []})
    for i, ax in enumerate(axes.flat):
        img = plt.imread(test_df['filename'].iloc[i])
        ax.imshow(img)
        t = true_classes[i]
        p = pred_classes[i]
        c = probs[i, preds[i]] * 100
        color = 'green' if t == p else 'red'
        ax.set_title('T: ' + t + '\nP: ' + p + ' (' + str(round(c)) + '%)',
                     color=color, fontsize=8)
    fig.suptitle(name + ' — Sample Predictions  (green=correct  red=wrong)', fontsize=11)
    plt.tight_layout()
    plt.show()

    return {
        'Model': name,
        'Test Loss': loss,
        'Test Accuracy': acc,
        'Macro F1': macro_f1,
        'Weighted F1': weighted_f1,
        'Params': model.count_params(),
        '_preds': pred_classes,
        '_trues': true_classes
    }

In [ ]:
# ── 4.1 AlexNet
test_gen_alex = make_test_gen(alexnet_preprocess)
result_alexnet = evaluate_model('AlexNet', model_alexnet, test_gen_alex)

In [ ]:
# ── 4.2 VGG16
test_gen_vgg = make_test_gen(vgg16_preprocess)
result_vgg = evaluate_model('VGG16', model_vgg, test_gen_vgg)

In [ ]:
# ── 4.3 ResNet50
test_gen_resnet = make_test_gen(resnet_preprocess)
result_resnet = evaluate_model('ResNet50', model_resnet, test_gen_resnet)

In [ ]:
# ── 4.4 MobileNetV2
test_gen_mobilenet = make_test_gen(mobilenet_preprocess)
result_mobilenet = evaluate_model('MobileNetV2', model_mobilenet, test_gen_mobilenet)

In [ ]:
# ── 4.5 ViT-B/16
test_gen_vit = make_test_gen(rescale)
result_vit = evaluate_model('ViT-B/16', model_vit, test_gen_vit)

## Section 5 — Cross-Model Comparison

In [ ]:
# ── 5.1 Summary Table
all_results = [result_alexnet, result_vgg, result_resnet, result_mobilenet, result_vit]

summary = pd.DataFrame([{
    'Model'         : r['Model'],
    'Test Loss'     : round(r['Test Loss'], 4),
    'Test Accuracy' : round(r['Test Accuracy'] * 100, 2),
    'Macro F1'      : round(r['Macro F1'], 4),
    'Weighted F1'   : round(r['Weighted F1'], 4),
    'Parameters'    : r['Params'],
} for r in all_results])

display(
    summary.style
    .set_caption('Model Comparison Summary')
    .highlight_max(subset=['Test Accuracy', 'Macro F1', 'Weighted F1'], color='#d4edda')
    .highlight_min(subset=['Test Loss'], color='#d4edda')
    .format({'Test Accuracy': '{:.2f}%', 'Parameters': '{:,}'})
)

In [ ]:
# ── 5.2 Bar Chart: Accuracy & Macro F1
model_names = [r['Model'] for r in all_results]
accuracies  = [r['Test Accuracy'] * 100 for r in all_results]
macro_f1s   = [r['Macro F1'] * 100      for r in all_results]

x = np.arange(len(model_names))
w = 0.35

fig, ax = plt.subplots(figsize=(11, 5))
bars1 = ax.bar(x - w/2, accuracies, w, label='Test Accuracy (%)', color='#4c72b0')
bars2 = ax.bar(x + w/2, macro_f1s,  w, label='Macro F1 (%)',      color='#dd8452')

for bar in list(bars1) + list(bars2):
    h = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, h + 0.3,
            str(round(h, 1)), ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(model_names)
ax.set_ylim(0, 115)
ax.set_ylabel('Score (%)')
ax.set_title('Test Accuracy vs Macro F1 by Model')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# ── 5.3 Per-Class F1 Heatmap
f1_matrix = []
for r in all_results:
    per_class = f1_score(r['_trues'], r['_preds'], labels=CLASS_NAMES, average=None)
    f1_matrix.append(per_class)

f1_df = pd.DataFrame(f1_matrix, index=model_names, columns=CLASS_NAMES)

fig, ax = plt.subplots(figsize=(max(8, len(CLASS_NAMES) * 1.2 + 2), 4))
sns.heatmap(f1_df, annot=True, fmt='.2f', cmap='YlGn',
            vmin=0, vmax=1, linewidths=0.5, ax=ax)
ax.set_title('Per-Class F1 Score — All Models')
ax.set_xlabel('Weather Class')
ax.set_ylabel('Model')
plt.tight_layout()
plt.show()

In [ ]:
# ── 5.4 Test Loss Comparison
losses  = [r['Test Loss'] for r in all_results]
colors  = ['#e74c3c' if l == min(losses) else '#3498db' for l in losses]

fig, ax = plt.subplots(figsize=(8, 4))
bars    = ax.bar(model_names, losses, color=colors, edgecolor='white', linewidth=0.8)
for bar, val in zip(bars, losses):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            str(round(val, 4)), ha='center', va='bottom', fontsize=9)
ax.set_ylabel('Test Loss')
ax.set_title('Test Loss by Model  (red = lowest = best)')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## Section 6 — Deeper Analysis

### 6.1 Grad-CAM Saliency Maps

In [ ]:
import cv2  # pip install opencv-python-headless

def make_gradcam_heatmap(img_array, model, last_conv_layer_name):
    grad_model = tf.keras.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(last_conv_layer_name).output, model.output]
    )
    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array, training=False)
        pred_index    = tf.argmax(predictions[0])
        class_channel = predictions[:, pred_index]
    grads         = tape.gradient(class_channel, conv_outputs)
    pooled_grads  = tf.reduce_mean(grads, axis=(0, 1, 2))
    heatmap       = conv_outputs[0] @ pooled_grads[..., tf.newaxis]
    heatmap       = tf.squeeze(heatmap)
    heatmap       = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy(), int(pred_index)

def overlay_gradcam(img_path, heatmap, alpha=0.4):
    img       = cv2.imread(img_path)
    img       = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img       = cv2.resize(img, IMAGE_SIZE)
    hm_res    = cv2.resize(heatmap, IMAGE_SIZE)
    hm_uint8  = np.uint8(255 * hm_res)
    hm_color  = cv2.applyColorMap(hm_uint8, cv2.COLORMAP_JET)
    hm_color  = cv2.cvtColor(hm_color, cv2.COLOR_BGR2RGB)
    overlay   = cv2.addWeighted(img, 1 - alpha, hm_color, alpha, 0)
    return img, overlay

print('Grad-CAM helpers defined.')

In [ ]:
# Pick best CNN (excluding ViT) and map to its last conv layer name
cnn_results = [r for r in all_results if r['Model'] != 'ViT-B/16']
best_cnn    = max(cnn_results, key=lambda r: r['Test Accuracy'])
print('Best CNN Model:', best_cnn['Model'])

# (model, preprocess_fn, last_conv_layer_name)
model_map = {
    'AlexNet'    : (model_alexnet,   alexnet_preprocess,   'conv5'),
    'VGG16'      : (model_vgg,       vgg16_preprocess,     'block5_conv3'),
    'ResNet50'   : (model_resnet,    resnet_preprocess,    'conv5_block3_out'),
    'MobileNetV2': (model_mobilenet, mobilenet_preprocess, 'block_16_project_BN'),
}

chosen_model, chosen_pre, last_conv = model_map[best_cnn['Model']]

fig, axes = plt.subplots(NUM_CLASSES, 2, figsize=(7, 3.5 * NUM_CLASSES))
if NUM_CLASSES == 1:
    axes = [axes]

for idx, cls in enumerate(CLASS_NAMES):
    sample_path = test_df[test_df['label'] == cls]['filename'].iloc[0]
    raw         = tf.keras.preprocessing.image.load_img(sample_path, target_size=IMAGE_SIZE)
    arr         = tf.keras.preprocessing.image.img_to_array(raw)
    arr_pre     = chosen_pre(arr.copy())
    inp         = arr_pre[np.newaxis, ...]
    heatmap, pred_idx = make_gradcam_heatmap(inp, chosen_model, last_conv)
    orig, overlay     = overlay_gradcam(sample_path, heatmap)

    axes[idx][0].imshow(orig)
    axes[idx][0].set_title('Original — ' + cls, fontsize=9)
    axes[idx][0].axis('off')
    axes[idx][1].imshow(overlay)
    axes[idx][1].set_title('Grad-CAM — Pred: ' + CLASS_NAMES[pred_idx], fontsize=9)
    axes[idx][1].axis('off')

fig.suptitle('Grad-CAM Saliency Maps — ' + best_cnn['Model'], fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

### 6.2 Error Analysis — Most Confident Mistakes

In [ ]:
best_overall = max(all_results, key=lambda r: r['Macro F1'])
print('Error analysis for:', best_overall['Model'])

trues  = best_overall['_trues']
preds  = best_overall['_preds']
errors = [(i, t, p) for i, (t, p) in enumerate(zip(trues, preds)) if t != p]

model_info = {
    'AlexNet'    : (model_alexnet,   test_gen_alex),
    'VGG16'      : (model_vgg,       test_gen_vgg),
    'ResNet50'   : (model_resnet,    test_gen_resnet),
    'MobileNetV2': (model_mobilenet, test_gen_mobilenet),
    'ViT-B/16'   : (model_vit,       test_gen_vit),
}
best_model_obj, best_gen = model_info[best_overall['Model']]
best_gen.reset()
all_probs = best_model_obj.predict(best_gen, verbose=0)

errors_conf = sorted(
    errors,
    key=lambda e: all_probs[e[0], CLASS_NAMES.index(e[2])],
    reverse=True
)

MAX_DISPLAY = min(12, len(errors_conf))
print('Total misclassified:', len(errors_conf), '/', len(trues))

if MAX_DISPLAY > 0:
    fig, axes = plt.subplots(3, 4, figsize=(14, 9),
                             subplot_kw={'xticks': [], 'yticks': []})
    for i, ax in enumerate(axes.flat):
        if i >= MAX_DISPLAY:
            ax.axis('off')
            continue
        idx, true_lbl, pred_lbl = errors_conf[i]
        conf = all_probs[idx, CLASS_NAMES.index(pred_lbl)] * 100
        img  = plt.imread(test_df['filename'].iloc[idx])
        ax.imshow(img)
        ax.set_title('True: ' + true_lbl + '\nPred: ' + pred_lbl +
                     ' (' + str(round(conf)) + '%)', color='red', fontsize=8)
    fig.suptitle('Most Confident Mistakes — ' + best_overall['Model'], fontsize=11)
    plt.tight_layout()
    plt.show()
else:
    print('No misclassified samples — perfect test accuracy!')

---
## 🏆 Final Leaderboard

In [ ]:
ranked = sorted(all_results, key=lambda r: r['Macro F1'], reverse=True)
medals = ['1st', '2nd', '3rd', '4th', '5th']
print('Model Leaderboard (ranked by Macro F1)\n')
header = '{:<5} {:<14} {:>11} {:>10} {:>13} {:>14}'.format(
    'Rank', 'Model', 'Accuracy', 'Macro F1', 'Weighted F1', 'Parameters')
print(header)
print('-' * 65)
for rank, r in enumerate(ranked, 1):
    print('{:<5} {:<14} {:>10.2f}% {:>10.4f} {:>13.4f} {:>14,}'.format(
        medals[rank-1], r['Model'],
        r['Test Accuracy'] * 100, r['Macro F1'],
        r['Weighted F1'], r['Params']
    ))